In [2]:
# for linux
# !apt-get install poppler-utils tesseract-ocr libmagic-dev

# for mac
# !brew install poppler tesseract libmagic

In [1]:
%pip install -Uq "unstructured[all-docs]" 
%pip install -Uq langchain_chroma 
%pip install pymupdf
%pip install -Uq langchain langchain-community langchain-openai 
%pip install -Uq python_dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

/Users/yifeishang/Desktop/desk/RAG/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
import fitz  # pip install pymupdf
import base64

def has_text_layer(pdf_path: str, sample_pages: int = 5) -> bool:
    """检查 PDF 是否有可提取的文字层（对比扫描版）"""
    doc = fitz.open(pdf_path)
    total = min(sample_pages, len(doc))
    char_count = sum(len(doc[i].get_text()) for i in range(total))
    doc.close()
    return char_count > 100 * total  # 平均每页超过100字符认为有文字层


def extract_titles_by_fontsize(pdf_path: str, min_font_size: float = 13.0) -> dict:
    """方案 B：通过字体大小提取每页标题，返回 {page_num: title}"""
    doc = fitz.open(pdf_path)
    page_titles = {}
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        blocks = page.get_text('dict')['blocks']
        candidates = []
        
        for block in blocks:
            for line in block.get('lines', []):
                for span in line.get('spans', []):
                    text = span['text'].strip()
                    size = span['size']
                    if size >= min_font_size and text and len(text) > 2:
                        candidates.append((size, text))
        
        if candidates:
            # 取字体最大的那个作为标题
            candidates.sort(key=lambda x: -x[0])
            page_titles[page_num + 1] = candidates[0][1]
    
    doc.close()
    return page_titles


def extract_title_by_vision(pdf_path: str, page_num: int) -> str:
    """方案 A：用 GPT-4o 视觉识别单页的产品名称"""
    doc = fitz.open(pdf_path)
    page = doc[page_num - 1]
    pix = page.get_pixmap(dpi=150)
    img_b64 = base64.b64encode(pix.tobytes('jpeg')).decode()
    doc.close()
    
    llm = ChatOpenAI(model='gpt-4o', temperature=0)
    msg = HumanMessage(content=[
        {"type": "text", "text": "This is a building materials product catalogue page. What is the main product name or heading on this page? Reply with just the product name, nothing else. If there is no clear product name, reply 'UNKNOWN'."},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
    ])
    return llm.invoke([msg]).content.strip()


def get_page_titles(pdf_path: str) -> dict:
    """自动判断：有文字层用方案B（字体大小），没有用方案A（GPT-4o视觉）"""
    print('🔍 Checking PDF text layer...')
    
    if has_text_layer(pdf_path):
        print('✅ Text layer found → using font size extraction (fast)')
        titles = extract_titles_by_fontsize(pdf_path)
    else:
        print('⚠️  No text layer → using GPT-4o vision (slower)')
        doc = fitz.open(pdf_path)
        titles = {}
        for page_num in range(1, len(doc) + 1):
            print(f'   Analyzing page {page_num}/{len(doc)}...')
            titles[page_num] = extract_title_by_vision(pdf_path, page_num)
        doc.close()
    
    print(f'✅ Extracted titles for {len(titles)} pages')
    return titles


# 运行：获取每页的标题
file_path = "./docs/2_Sikafloor-263 SL.pdf"
page_titles = get_page_titles(file_path)

# 查看结果（只打印有内容的页）
for page, title in sorted(page_titles.items()):
    print(f'  Page {page:3d}: {title}')

🔍 Checking PDF text layer...
✅ Text layer found → using font size extraction (fast)
✅ Extracted titles for 5 pages
  Page   1: Sikafloor®-263 SL
  Page   2: PRODUCT INFORMATION
  Page   3: SYSTEM INFORMATION
  Page   4: BASIS OF PRODUCT DATA
  Page   5: MAINTENANCE


In [4]:
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=[], # Grab images found in the PDF
        extract_image_block_to_payload=False # Store images as base64 data you can actually use
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

# Test with your PDF file
  # Change this to your PDF path
elements = partition_document(file_path)

📄 Partitioning document: ./docs/2_Sikafloor-263 SL.pdf


Loading weights: 100%|██████████| 367/367 [00:00<00:00, 1604.67it/s, Materializing param=model.query_position_embeddings.weight]                             


✅ Extracted 141 elements


In [ ]:
# All types of different atomic elements we see from unstructured
# set([str(type(el)) for el in elements])

In [7]:
def create_chunks_by_title(elements, page_titles: dict = None):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    
    # 提取构件名和对应表格
    components = []
    for i, chunk in enumerate(chunks):
        component_name = None
        chunk_pages = set()
        tables = []
        if hasattr(chunk.metadata, 'orig_elements'):
            for el in chunk.metadata.orig_elements:
                el_type = type(el).__name__
                if el_type == 'Title':
                    component_name = el.text.strip()  # 每次遇到 Title 都覆盖，最终保留最后一个
                elif el_type == 'Table':
                    tables.append(getattr(el.metadata, 'text_as_html', el.text))
                # 收集这个 chunk 涉及的页码
                page = getattr(el.metadata, 'page_number', None)
                if page:
                    chunk_pages.add(page)
        # 如果没有 Title，用 page_titles 补全（取 chunk 第一页的标题）
        if not component_name and page_titles and chunk_pages:
            first_page = min(chunk_pages)
            component_name = page_titles.get(first_page)
        if component_name:
            components.append({'chunk_id': i, 'component_name': component_name, 'tables': tables})
    
    print(f"✅ Found {len(components)} components")
    for c in components:
        print(f"  [{c['chunk_id']}] {c['component_name']} — {len(c['tables'])} 张表")
    
    return chunks, components

# Create chunks，传入 page_titles 补全被 OCR 漏掉的标题
chunks, components = create_chunks_by_title(elements, page_titles=page_titles)

🔨 Creating smart chunks...
✅ Created 15 chunks
✅ Found 15 components
  [0] SUSTAINABILITY — 0 张表
  [1] CHARACTERISTICS / ADVANTAGES — 0 张表
  [2] 1 / — 2 张表
  [3] APPLICATION INFORMATION — 2 张表
  [4] 3 / — 2 张表
  [5] FURTHER INFORMATION — 0 张表
  [6] ECOLOGY, HEALTH AND SAFETY — 0 张表
  [7] Mixing Tools — 0 张表
  [8] SUBSTRATE QUALITY / PRE-TREATMENT — 0 张表
  [9] APPLICATION — 0 张表
  [10] Levelling: — 0 张表
  [11] CLEANING OF EQUIPMENT — 0 张表
  [12] LOCAL RESTRICTIONS — 0 张表
  [13] LEGAL NOTES — 0 张表
  [14] Sika Saudi Arabia — 0 张表


In [ ]:
import fitz
import base64
import json
import os

def page_to_base64(pdf_path: str, page_num: int, dpi: int = 150) -> str:
    """Convert a PDF page to base64 JPEG image"""
    doc = fitz.open(pdf_path)
    page = doc[page_num - 1]
    pix = page.get_pixmap(dpi=dpi)
    img_b64 = base64.b64encode(pix.tobytes('jpeg')).decode()
    doc.close()
    return img_b64


def extract_page_catalogue(pdf_path: str, page_num: int, llm) -> dict:
    """Ask GPT-4o to extract product catalogue data from a single page"""
    img_b64 = page_to_base64(pdf_path, page_num)
    
    prompt = """This is a page from a building materials product catalogue.

Extract ALL product data from this page and return it as JSON with this structure:
{
  \"product_name\": \"ANGLE BEAD\",
  \"description\": \"brief product description\",
  \"items\": [
    {\"item_number\": \"SMAB_F_00054403\", \"attribute1\": \"value1\", \"attribute2\": \"value2\"},
    ...
  ]
}

Rules:
- Use the exact column names from the table as attribute keys
- If the page has no product table, return {\"product_name\": null, \"items\": []}
- Return ONLY valid JSON, no extra text"""

    msg = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
    ])
    
    response = llm.invoke([msg]).content.strip()
    if response.startswith('```'):
        response = response.split('```')[1]
        if response.startswith('json'):
            response = response[4:]
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        return {"product_name": None, "items": []}


def build_catalogue(pdf_path: str) -> dict:
    """逐页用 GPT-4o 提取产品目录。当某页没有标题时，沿用上一页的类目"""
    llm = ChatOpenAI(model='gpt-4o', temperature=0)
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    doc.close()

    catalogue = {}
    last_product_name = None
    print(f'Processing {total_pages} pages...')

    for page_num in range(1, total_pages + 1):
        print(f'  Page {page_num}/{total_pages}...', end=' ')
        result = extract_page_catalogue(pdf_path, page_num, llm)

        product_name = result.get('product_name')
        items = result.get('items', [])

        if not product_name and items and last_product_name:
            product_name = last_product_name
            print(f'\u21a9\ufe0f  沿用上一类目: {product_name} ({len(items)} items)', end=' ')

        if product_name and items:
            if product_name not in catalogue:
                catalogue[product_name] = {'description': result.get('description', ''), 'items': []}
            existing_ids = {item.get('item_number') for item in catalogue[product_name]['items']}
            new_items = [item for item in items if item.get('item_number') not in existing_ids]
            catalogue[product_name]['items'].extend(new_items)
            last_product_name = product_name
            print(f'\u2705 {product_name} ({len(new_items)} items)')
        else:
            print('skipped')

    print(f'\n\u2705 Done! Found {len(catalogue)} products')
    return catalogue

gpt_catalogue = build_catalogue(file_path)

for product_name, data in gpt_catalogue.items():
    print(f'\n=== {product_name} ({len(data["items"])} items) ===')
    print(f'  Description: {data["description"]}')
    for item in data['items']:
        print(f'  {item}')

📂 恢复进度：已完成 33/5 页，上一个类目：ANGLE BEAD

✅ Done! Found 16 products

=== ANGLE BEAD (197 items) ===
  Description: Angle beads provide with its solid metal nose a straight corner. Expanded diamond mesh wings allow for keying the plaster right up to the nose of the bead. It is designed to protect the corners. The flanges can be easily fixed over irregular, uneven surfaces. Guarantees a perfect bond and provides better effective reinforcement at corners where it is mostly needed. Angle bead is recommended for a greater corner protection and a precise straight line. Available in galvanized finish for internal use, and in stainless steel for external use.
  {'item_number': 'SMAB_F_00054403', 'Length of wings (mm)': '50', 'Plaster Depth (mm)': '12 - 19', 'Length (mm)': '2700', 'Material': 'GI'}
  {'item_number': 'SMAB_F_00054405', 'Length of wings (mm)': '50', 'Plaster Depth (mm)': '12 - 19', 'Length (mm)': '2850', 'Material': 'GI'}
  {'item_number': 'SMAB_F_00054411', 'Length of wings (mm)': '50

In [9]:
import json, os, threading, webbrowser
from http.server import HTTPServer, SimpleHTTPRequestHandler

# 1. Export catalogue to JSON (re-run anytime to refresh data)
with open('catalogue_data.json', 'w', encoding='utf-8') as f:
    json.dump(gpt_catalogue, f, indent=2, ensure_ascii=False)
print('✅ Exported catalogue_data.json')

# 2. Start local HTTP server (needed so fetch() can load the JSON)
PORT = 8765

class SilentHandler(SimpleHTTPRequestHandler):
    def log_message(self, format, *args): pass  # suppress server logs

def start_server():
    os.chdir(os.path.dirname(os.path.abspath('frontend.html')))
    server = HTTPServer(('localhost', PORT), SilentHandler)
    server.serve_forever()

# Only start if not already running
try:
    import socket
    s = socket.socket()
    s.connect(('localhost', PORT))
    s.close()
    print(f'✅ Server already running on port {PORT}')
except ConnectionRefusedError:
    t = threading.Thread(target=start_server, daemon=True)
    t.start()
    print(f'✅ Server started on port {PORT}')

# 3. Open browser
url = f'http://localhost:{PORT}/frontend.html'
webbrowser.open(url)
print(f'🌐 Opening {url}')
print('   (Re-run this cell after updating catalogue_data.json to reload data)')

✅ Exported catalogue_data.json
✅ Server already running on port 8765
🌐 Opening http://localhost:8765/frontend.html
   (Re-run this cell after updating catalogue_data.json to reload data)
